hecfda ports the numerical core of USACE HEC-FDA to C++ with R and Python packages on top. This example installs the package, draws from the seeded generator, and evaluates a distribution. The draws below are identical in R, Python, and the upstream C#, because all three run the same algorithm: .NET's seeded `System.Random`.


## Install

```bash
pip install "git+https://github.com/cameronbracken/hecfda.git#subdirectory=hecfdapy"
```


## Seeded reproducibility


In [ ]:
import hecfdapy as fda

fda.rng_sequence(seed=1234, n=5)


The same call in R (`rng_sequence(seed = 1234L, n = 5L)`) returns the same five numbers, and so does `new Random(1234).NextDouble()` in .NET. Every Monte Carlo compute in hecfda draws from this generator with fixed per-curve seeds, so full simulation results carry the same guarantee.


## A first distribution


In [ ]:
fda.dist_cdf("Normal", [0, 1, 1], 0.0)  # 0.5


In [ ]:
# the "100-year" quantile
fda.dist_quantile("LogPearsonIII", [3.3, 0.254, 0.0925, 99], 1 - 0.01)


## A first EAD

The smallest useful compute: a uniform flow frequency, a linear flow-stage transform, one stage-damage curve, one deterministic pass. This reproduces the HEC-FDA deterministic oracle of 150000 (the mean of the 600000-damage plateau over the frequency curve).


In [ ]:
res = fda.ead_simulation(
    stage_damage=[{
        "x": [0, 15, 30], "dist": "Uniform",
        "params": [[0, 0, 10], [0, 600000, 10], [0, 600000, 10]],
        "damage_category": "residential", "asset_category": "Structure",
    }],
    flow_frequency={"dist": "Uniform", "params": [0, 100000, 1000]},
    flow_stage={"x": [0, 100000], "dist": "Uniform", "params": [[0, 0, 10], [0, 30, 10]]},
    min_iterations=1, max_iterations=1, deterministic=True,
)
res["total_ead"]
